# WorldQuant BRAIN Alpha 遍历 V2

正式运行流程：Data Fields → Stage 1 → Promotion → Stage 2 → Targeted Repair → Final Candidates → Submission Check。

- SQLite 缓存与自动断点续跑
- COMPLETE 不重复回测
- SUBMITTED/RUNNING 从已有 simulation URL 恢复
- ERROR 自动重试

登录凭据由 `machine_lib.py` 从项目目录 TXT 自动读取。


## 1. Environment

加载并刷新项目模块，确认当前代码与项目路径。


In [37]:
import sys
import importlib
from pathlib import Path

import pandas as pd
import machine_lib

machine_lib = importlib.reload(machine_lib)
from machine_lib import *

module_path = Path(machine_lib.__file__).resolve()
project_dir = module_path.parent

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)

print("machine_lib:", module_path)
print("Project:", project_dir)
print("Python:", sys.executable)


machine_lib: F:\一二三\v2\machine_lib.py
Project: F:\一二三\v2
Python: c:\Users\Administrator\AppData\Local\Programs\Python\Python39\python.exe


## 2. Configuration

生产设置与真实回测缓存键保持一致。所有联网执行开关默认关闭。


In [38]:
REGION = "GBR"
UNIVERSE = "TOP700"
DELAY = 1
DATASET_ID = "other169"

NEUTRALIZATION = "SUBINDUSTRY"
INIT_DECAY = 4
TRUNCATION = 0.08
# 最大同时活跃的simulation数量。
# 1=串行；修改该值即可控制Stage1/Stage2/Repair并发度。
CONCURRENCY = 6
MAX_FIELDS = None

CACHE_DB = str(project_dir / "alpha_results.db")

RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_REPAIR = True
RUN_SUBMISSION_CHECK = False

print(REGION, UNIVERSE, DATASET_ID, NEUTRALIZATION, INIT_DECAY, TRUNCATION)
print("CACHE_DB:", CACHE_DB)


GBR TOP700 other169 SUBINDUSTRY 4 0.08
CACHE_DB: F:\一二三\v2\alpha_results.db


## 3. Login & Data Fields

登录 BRAIN，读取当前 Dataset，并准备可用于候选生成的字段表达式。


In [ ]:
s = login()
print("BRAIN login: OK")


In [39]:
df = get_datafields(
    s,
    dataset_id=DATASET_ID,
    region=REGION,
    universe=UNIVERSE,
    delay=DELAY,
)

field_records = prepare_fields(df)
if MAX_FIELDS is not None:
    field_records = field_records[:MAX_FIELDS]

print("Data fields:", len(df))
print("Prepared field expressions:", len(field_records))
display(df.head(5))
display(pd.DataFrame(field_records).head(5))


Data fields: 22
Prepared field expressions: 44


,id,description,dataset,category,subcategory,region,delay,universe,type,dateCoverage,coverage,userCount,alphaCount,pyramidMultiplier,themes,dateCreated
0,bid_par_spread_basis_points,Bid (best buyer price) for CDS par spread (annualized premium in basis points),"{'id': 'other169', 'name': 'CDS dataset'}","{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Dividend Models'}",GBR,1,TOP700,VECTOR,1.0000,0.0,0,0,1.9,[],2026-07-01
1,bid_price_percent_of_notional,Bid price expressed as percent of par (notional) amount,"{'id': 'other169', 'name': 'CDS dataset'}","{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Dividend Models'}",GBR,1,TOP700,VECTOR,0.6418,0.0,0,0,1.9,[],2026-07-01
2,bid_quote_spread_basis_points,Bid for the CDS quote spread vs coupon in basis points,"{'id': 'other169', 'name': 'CDS dataset'}","{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Dividend Models'}",GBR,1,TOP700,VECTOR,1.0000,0.0,0,0,1.9,[],2026-07-01
3,bid_upfront_payment_percent,Bid (buyer price) for upfront payment (as percent of notional) to enter the CDS contract,"{'id': 'other169', 'name': 'CDS dataset'}","{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Dividend Models'}",GBR,1,TOP700,VECTOR,1.0000,0.0,0,0,1.9,[],2026-07-01
4,highest_mid_value_contributed,Highest mid price/spread value observed or contributed during the trading day,"{'id': 'other169', 'name': 'CDS dataset'}","{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Dividend Models'}",GBR,1,TOP700,VECTOR,1.0000,0.0,0,0,1.9,[],2026-07-01


,field,field_type,vector_op,base_expr,expr,stage,operator,window,parent,decay
0,bid_par_spread_basis_points,VECTOR,vec_avg,vec_avg(bid_par_spread_basis_points),"winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4)",0,preprocess,120,None,6
1,bid_par_spread_basis_points,VECTOR,vec_sum,vec_sum(bid_par_spread_basis_points),"winsorize(ts_backfill(vec_sum(bid_par_spread_basis_points), 120), std=4)",0,preprocess,120,None,6
2,bid_price_percent_of_notional,VECTOR,vec_avg,vec_avg(bid_price_percent_of_notional),"winsorize(ts_backfill(vec_avg(bid_price_percent_of_notional), 120), std=4)",0,preprocess,120,None,6
3,bid_price_percent_of_notional,VECTOR,vec_sum,vec_sum(bid_price_percent_of_notional),"winsorize(ts_backfill(vec_sum(bid_price_percent_of_notional), 120), std=4)",0,preprocess,120,None,6
4,bid_quote_spread_basis_points,VECTOR,vec_avg,vec_avg(bid_quote_spread_basis_points),"winsorize(ts_backfill(vec_avg(bid_quote_spread_basis_points), 120), std=4)",0,preprocess,120,None,6


## 4. Stage 1 Candidates

使用当前 V2 搜索空间生成第一阶段候选；本节不会创建 simulation。


In [40]:
stage1_candidates = first_order_candidates(
    field_records,
    ts_operators=CORE_TS_OPS,
    cross_ops=("rank", "zscore"),
    init_decay=INIT_DECAY,
)

stage1_unique_count = len({candidate["expr"] for candidate in stage1_candidates})
print("Stage 1 candidates:", len(stage1_candidates))
print("Unique expressions:", stage1_unique_count)

stage1_preview = pd.DataFrame(stage1_candidates)
stage1_preview_columns = [
    column for column in ("field", "operator", "window", "decay", "expr")
    if column in stage1_preview.columns
]
display(stage1_preview[stage1_preview_columns].head(10))


Stage 1 candidates: 748
Unique expressions: 748


,field,operator,window,decay,expr
0,bid_par_spread_basis_points,raw,NaN,4,"winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4)"
1,bid_par_spread_basis_points,rank,NaN,4,"rank(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4))"
2,bid_par_spread_basis_points,zscore,NaN,4,"zscore(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4))"
3,bid_par_spread_basis_points,ts_rank,22.0,4,"ts_rank(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 22)"
4,bid_par_spread_basis_points,ts_rank,66.0,4,"ts_rank(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 66)"
5,bid_par_spread_basis_points,ts_rank,120.0,4,"ts_rank(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 120)"
6,bid_par_spread_basis_points,ts_zscore,22.0,4,"ts_zscore(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 22)"
7,bid_par_spread_basis_points,ts_zscore,66.0,4,"ts_zscore(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 66)"
8,bid_par_spread_basis_points,ts_zscore,120.0,4,"ts_zscore(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 120)"
9,bid_par_spread_basis_points,ts_delta,5.0,4,"ts_delta(winsorize(ts_backfill(vec_avg(bid_par_spread_basis_points), 120), std=4), 5)"


## 5. Stage 1 Status

本 Cell 只读取 SQLite 状态，不会创建 simulation。


In [41]:
cache_summary(CACHE_DB)

stage1_resume = resume_summary(
    stage1_candidates,
    neutralization=NEUTRALIZATION,
    region=REGION,
    universe=UNIVERSE,
    cache_db=CACHE_DB,
    delay=DELAY,
    truncation=TRUNCATION,
    test_period="P0Y",
)


Cache summary
-------------
TOTAL: 1362
COMPLETE: 1357
SUBMITTED: 0
RUNNING: 2
ERROR: 1
INVALID: 0
AUTH_ERROR: 2
UNCERTAIN_SUBMISSION: 0
Resume summary
Candidates: 748
Unique: 748

Completed cache: 0
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 748

Remaining simulations: 748


## 6. Run Stage 1

设置 `RUN_STAGE1=True` 后运行；中断后重新运行本 Cell 即可续跑。


In [44]:
if RUN_STAGE1:
    stage1_results = simulate_candidates(
        stage1_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period="P0Y",
        progress_every=10,
    )
else:
    stage1_results = pd.DataFrame()
    print("Stage 1 未启动：RUN_STAGE1=False")


Resume summary
Candidates: 748
Unique: 748

Completed cache: 386
Submitted / running: 7
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 355

Remaining simulations: 362
Simulation concurrency: 6
[001/748] CACHE COMPLETE
[002/748] CACHE COMPLETE
[003/748] CACHE COMPLETE
[004/748] CACHE COMPLETE
[005/748] CACHE COMPLETE
[006/748] CACHE COMPLETE
[007/748] CACHE COMPLETE
[008/748] CACHE COMPLETE
[009/748] CACHE COMPLETE
[010/748] CACHE COMPLETE
[011/748] CACHE COMPLETE
[012/748] CACHE COMPLETE
[013/748] CACHE COMPLETE
[014/748] CACHE COMPLETE
[015/748] CACHE COMPLETE
[016/748] CACHE COMPLETE
[017/748] CACHE COMPLETE
[018/748] CACHE COMPLETE
[019/748] CACHE COMPLETE
[020/748] CACHE COMPLETE
[021/748] CACHE COMPLETE
[022/748] CACHE COMPLETE
[023/748] CACHE COMPLETE
[024/748] CACHE COMPLETE
[025/748] CACHE COMPLETE
[026/748] CACHE COMPLETE
[027/748] CACHE COMPLETE
[028/748] CACHE COMPLETE
[029/748] CACHE COMPLETE
[030/748] CACHE COMPLETE
[0

## 7. Stage 1 Results

汇总 Stage 1 终态、显示 Top 20，并按固定阈值生成 Stage 2 晋级集合。


In [45]:
stage1_scored = pd.DataFrame()
stage1_promoted = []
stage1_selected = pd.DataFrame()

if stage1_results.empty:
    print("暂无 Stage 1 results。")
else:
    stage1_scored = score_results(stage1_results)
    stage1_status = stage1_scored["status"].fillna("ERROR").astype(str).str.upper()
    stage1_complete_count = int((stage1_status == "COMPLETE").sum())

    print("TOTAL:", len(stage1_scored))
    for status_name in ("COMPLETE", "ERROR", "INVALID"):
        print(f"{status_name}:", int((stage1_status == status_name).sum()))

    stage1_top_columns = [
        column for column in (
            "alpha_id", "field", "operator", "window", "sharpe", "fitness",
            "turnover", "margin", "positions", "score"
        ) if column in stage1_scored.columns
    ]
    display(
        stage1_scored.sort_values("score", ascending=False)[stage1_top_columns].head(20)
    )

    stage1_promoted, stage1_selected = promote_candidates(
        stage1_results,
        min_abs_sharpe=0.80,
        min_abs_fitness=0.45,
        min_positions=100,
        keep_per_field=2,
        max_total=60,
    )

    promotion_rate = len(stage1_promoted) / stage1_complete_count if stage1_complete_count else 0.0
    print("Stage 1 COMPLETE:", stage1_complete_count)
    print("Stage 1 promoted:", len(stage1_promoted))
    print("Promotion rate:", f"{promotion_rate:.2%}")


TOTAL: 748
COMPLETE: 743
ERROR: 3
INVALID: 0


,alpha_id,sharpe,fitness,turnover,margin,positions,score
235,58pzJzJX,0.84,0.58,0.0229,0.005106,162.0,0.933957
234,O0GNWqnY,0.86,0.60,0.0251,0.004813,161.0,0.933322
132,LLGNN5j1,-1.02,-1.13,0.0255,-0.012128,125.0,0.927206
233,WjAblKMQ,0.84,0.58,0.0296,0.004052,161.0,0.926404
223,58pzE1E1,0.83,0.57,0.0338,0.003501,161.0,0.921725
221,LLGNMr06,0.83,0.57,0.0339,0.003494,161.0,0.921524
224,d5ZbWJaX,1.81,0.99,0.3260,0.000601,160.0,0.919118
404,88pjZZX7,-0.99,-1.09,0.0258,-0.011764,124.0,0.919118
121,JjGNN0oW,-1.00,-1.11,0.0320,-0.009535,123.0,0.913803
237,d5Zbaxzw,0.74,0.57,0.0524,0.002805,162.0,0.913503


Stage 1 COMPLETE: 743
Stage 1 promoted: 17
Promotion rate: 2.29%


## 8. Stage 2 Candidates

对 Stage 1 晋级结果应用固定的核心 Group 搜索空间，并显示可恢复任务摘要。


In [48]:
if stage1_promoted:
    stage2_candidates = second_order_candidates(
        stage1_promoted,
        region=REGION,
        group_ops=("group_neutralize", "group_rank"),
        extended_groups=False,
    )
else:
    stage2_candidates = []

print("Stage 2 candidates:", len(stage2_candidates))
if stage2_candidates:
    stage2_preview = pd.DataFrame(stage2_candidates)
    stage2_preview_columns = [
        column for column in ("field", "group_operator", "group", "decay", "expr")
        if column in stage2_preview.columns
    ]
    display(stage2_preview[stage2_preview_columns].head(10))
    stage2_resume = resume_summary(
        stage2_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period="P0Y",
    )
else:
    stage2_resume = None
    print("没有 Stage 1 promoted candidates。")


Stage 2 candidates: 40


,field,group_operator,group,decay,expr
0,bid_upfront_payment_percent,group_neutralize,sector,4,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(s..."
1,bid_upfront_payment_percent,group_neutralize,industry,4,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(i..."
2,bid_upfront_payment_percent,group_neutralize,subindustry,4,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(s..."
3,bid_upfront_payment_percent,group_neutralize,"bucket(rank(cap), range='0.1, 1, 0.1')",4,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(b..."
4,bid_upfront_payment_percent,group_neutralize,"bucket(rank(close*volume), range='0.1, 1, 0.1')",4,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(b..."
5,bid_upfront_payment_percent,group_rank,sector,4,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(sector))"
6,bid_upfront_payment_percent,group_rank,industry,4,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(industry))"
7,bid_upfront_payment_percent,group_rank,subindustry,4,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(subindu..."
8,bid_upfront_payment_percent,group_rank,"bucket(rank(cap), range='0.1, 1, 0.1')",4,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(bucket(..."
9,bid_upfront_payment_percent,group_rank,"bucket(rank(close*volume), range='0.1, 1, 0.1')",4,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(bid_upfront_payment_percent), 120), std=4), 22)), densify(bucket(..."


Resume summary
Candidates: 40
Unique: 40

Completed cache: 0
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 40

Remaining simulations: 40


## 9. Run Stage 2

Stage 2 只有在 `RUN_STAGE2=True` 且存在候选时才会创建或恢复 simulation。


In [53]:
if RUN_STAGE2:
    if not stage2_candidates:
        stage2_results = pd.DataFrame()
        print("没有 Stage 1 promoted candidates，Stage 2 未运行。")
    else:
        stage2_results = simulate_candidates(
            stage2_candidates,
            neutralization=NEUTRALIZATION,
            region=REGION,
            universe=UNIVERSE,
            session=s,
            cache_db=CACHE_DB,
            concurrency=CONCURRENCY,
            delay=DELAY,
            truncation=TRUNCATION,
            test_period="P0Y",
            progress_every=10,
        )
else:
    stage2_results = pd.DataFrame()
    print("Stage 2 未启动：RUN_STAGE2=False")


Resume summary
Candidates: 40
Unique: 40

Completed cache: 40
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 0

Remaining simulations: 0
Simulation concurrency: 6
[001/40] CACHE COMPLETE
[002/40] CACHE COMPLETE
[003/40] CACHE COMPLETE
[004/40] CACHE COMPLETE
[005/40] CACHE COMPLETE
[006/40] CACHE COMPLETE
[007/40] CACHE COMPLETE
[008/40] CACHE COMPLETE
[009/40] CACHE COMPLETE
[010/40] CACHE COMPLETE
[011/40] CACHE COMPLETE
[012/40] CACHE COMPLETE
[013/40] CACHE COMPLETE
[014/40] CACHE COMPLETE
[015/40] CACHE COMPLETE
[016/40] CACHE COMPLETE
[017/40] CACHE COMPLETE
[018/40] CACHE COMPLETE
[019/40] CACHE COMPLETE
[020/40] CACHE COMPLETE
[021/40] CACHE COMPLETE
[022/40] CACHE COMPLETE
[023/40] CACHE COMPLETE
[024/40] CACHE COMPLETE
[025/40] CACHE COMPLETE
[026/40] CACHE COMPLETE
[027/40] CACHE COMPLETE
[028/40] CACHE COMPLETE
[029/40] CACHE COMPLETE
[030/40] CACHE COMPLETE
[031/40] CACHE COMPLETE
[032/40] CACHE 

## 10. Stage 2 Results

汇总 Stage 2 结果，并按固定阈值生成 Targeted Repair 的输入集合。


In [56]:
print("Stage 2 晋级数量:", len(stage2_promoted))
print("Stage 2 selected 数量:", len(stage2_selected))

if len(stage2_selected):
    display(
        stage2_selected[
            [
                "alpha_id",
                "field",
                "sharpe",
                "fitness",
                "turnover",
                "margin",
                "positions",
                "score",
            ]
        ].head(40)
    )
else:
    print("没有 Stage 2 晋级 Alpha")

Stage 2 晋级数量: 24
Stage 2 selected 数量: 24


,alpha_id,field,sharpe,fitness,turnover,margin,positions,score
2,ak1bk7a2,leg2_spread_value,1.73,3.04,0.0557,0.013850,481,0.917469
1,GrGbre93,leg2_spread_value,1.73,3.03,0.0568,0.013557,482,0.917050
30,VkGa1LK5,leg2_aggregate_notional_quantity,1.48,1.09,0.0527,0.002587,582,0.879393
52,QPGbMbvw,leg1_spread_value,1.70,1.90,0.0903,0.003459,104,0.878452
50,QPGbMNpp,leg1_spread_value,1.70,1.90,0.0991,0.003151,104,0.875314
31,88pj5RVX,leg2_aggregate_notional_quantity,1.47,1.08,0.0526,0.002579,582,0.875105
22,1YpXkodk,leg2_floating_reset_period_count,1.43,1.15,0.0770,0.002096,588,0.873849
21,QPGbZl2Q,leg2_floating_reset_period_count,1.42,1.14,0.0775,0.002074,588,0.868828
203,pwNR0V5g,price_expression_format,1.69,1.46,0.1270,0.001496,342,0.850628
82,ZYEb5NjQ,leg1_floating_reset_period_count,1.61,1.41,0.1099,0.001743,100,0.837448


In [ ]:
stage2_scored = pd.DataFrame()
stage2_promoted = []
stage2_selected = pd.DataFrame()

if stage2_results.empty:
    print("暂无 Stage 2 results。")
else:
    stage2_scored = score_results(stage2_results)
    stage2_top_columns = [
        column for column in (
            "alpha_id", "field", "operator", "window", "sharpe", "fitness",
            "turnover", "margin", "positions", "score"
        ) if column in stage2_scored.columns
    ]
    display(
        stage2_scored.sort_values("score", ascending=False)[stage2_top_columns].head(20)
    )

    stage2_promoted, stage2_selected = promote_candidates(
        stage2_results,
        min_abs_sharpe=1.00,
        min_abs_fitness=0.60,
        min_positions=100,
        keep_per_field=2,
        max_total=40,
    )
    print("Stage 2 promoted:", len(stage2_promoted))


## 11. Targeted Repair

仅为 Stage 2 入选结果生成定向修复候选；运行仍受独立安全门控制。


In [ ]:
if not stage2_selected.empty:
    repair_candidates = targeted_repair_candidates(
        stage2_selected,
        region=REGION,
        max_variants_per_parent=6,
        turnover_trigger=0.35,
    )
else:
    repair_candidates = []

print("Repair candidates:", len(repair_candidates))
if repair_candidates:
    repair_preview = pd.DataFrame(repair_candidates)
    repair_preview_columns = [
        column for column in ("field", "repair", "decay", "expr")
        if column in repair_preview.columns
    ]
    display(repair_preview[repair_preview_columns].head(10))
    repair_resume = resume_summary(
        repair_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period="P0Y",
    )
else:
    repair_resume = None
    print("没有 Stage 2 selected candidates。")


In [ ]:
if RUN_REPAIR:
    if not repair_candidates:
        repair_results = pd.DataFrame()
        print("没有 Stage 2 selected candidates，Repair 未运行。")
    else:
        repair_results = simulate_candidates(
            repair_candidates,
            neutralization=NEUTRALIZATION,
            region=REGION,
            universe=UNIVERSE,
            session=s,
            cache_db=CACHE_DB,
            concurrency=CONCURRENCY,
            delay=DELAY,
            truncation=TRUNCATION,
            test_period="P0Y",
            progress_every=10,
        )
else:
    repair_results = pd.DataFrame()
    print("Repair 未启动：RUN_REPAIR=False")


## 12. Final Candidates

合并 Stage 2 优秀结果与 Repair 结果，保留 COMPLETE 并按 Alpha ID 去重。


In [ ]:
final_frames = []

if not stage2_selected.empty:
    final_frames.append(stage2_selected)

if not repair_results.empty:
    final_frames.append(score_results(repair_results))

if final_frames:
    final_results = pd.concat(final_frames, ignore_index=True, sort=False)
    final_results = score_results(final_results)
    final_results = (
        final_results[final_results["status"].astype(str).str.upper() == "COMPLETE"]
        .sort_values(["score", "abs_sharpe"], ascending=False)
        .drop_duplicates(subset=["alpha_id"])
        .reset_index(drop=True)
    )
else:
    final_results = pd.DataFrame()

print("Final candidates:", len(final_results))
if not final_results.empty:
    final_columns = [
        column for column in (
            "alpha_id", "field", "sharpe", "fitness", "turnover",
            "margin", "positions", "score"
        ) if column in final_results.columns
    ]
    display(final_results[final_columns].head(30))


## 13. Submission Check

仅执行候选检查，不自动提交 Alpha；默认关闭。


In [ ]:
FINAL_CHECK_N = 30
stone_bag = []
gold_bag = []

if RUN_SUBMISSION_CHECK:
    if final_results.empty:
        print("暂无最终候选。")
    else:
        stone_bag = (
            final_results["alpha_id"]
            .dropna()
            .astype(str)
            .head(FINAL_CHECK_N)
            .tolist()
        )
        gold_bag = check_submission(stone_bag, [], 0)
        print("通过所有 check 的数量:", len(gold_bag))
        if gold_bag:
            view_alphas(gold_bag)
else:
    print("Submission Check 未启动。")


## 14. Export

SQLite 是 simulation 缓存与事实来源；非空结果另行导出到项目 `results` 目录。


In [ ]:
OUTPUT_DIR = project_dir / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

exports = {
    "stage1_results.csv": stage1_results,
    "stage2_results.csv": stage2_results,
    "repair_results.csv": repair_results,
    "final_results.csv": final_results,
}

for filename, result_frame in exports.items():
    if not result_frame.empty:
        output_path = OUTPUT_DIR / filename
        result_frame.to_csv(output_path, index=False)
        print("Exported:", output_path)


## 15. Maintenance

查看当前 SQLite 缓存状态。


In [ ]:
cache_summary(CACHE_DB)
